# Análisis Exploratorio de Datos — Airbnb Madrid vs Milán

Este notebook realiza un análisis exploratorio comparativo de los datos de Airbnb para **Madrid** y **Milán**.

## 1. Importar bibliotecas

Cargamos las librerías necesarias para el análisis:
- **pandas** y **numpy**: manipulación y cálculo con datos
- **matplotlib** y **seaborn**: visualización de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configuración general de gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.float_format', '{:.2f}'.format)

print(' Librerías cargadas correctamente')

## 2. Cargar los datos

Leemos los dos archivos CSV desde la carpeta `data/raw/`. Además añadimos una columna `ciudad` a cada dataset para poder distinguirlos cuando los unamos en un único DataFrame combinado.

In [ ]:
madrid = pd.read_csv('../data/raw/madrid_airbnb.csv')
milan  = pd.read_csv('../data/raw/milan_airbnb.csv')

# Añadimos una columna para identificar la ciudad
madrid['ciudad'] = 'Madrid'
milan['ciudad']  = 'Milán'

# Dataset combinado
df = pd.concat([madrid, milan], ignore_index=True)

print(f'Madrid: {madrid.shape}  |  Milán: {milan.shape}  |  Total: {df.shape}')

## 3. Explorar las primeras filas

Un primer vistazo a las filas iniciales de cada dataset nos permite ver el formato de los datos, los nombres de las columnas y los valores típicos. Esto es útil para detectar rápidamente problemas evidentes como columnas mal nombradas o valores inesperados.

In [ ]:
print('=== MADRID ===')
display(madrid.head())
print('\n=== MILÁN ===')
display(milan.head())

## 4. Información general del dataset

Con `info()` obtenemos el número de filas, columnas, el tipo de dato de cada columna y cuántos valores no nulos hay. Es el primer paso para detectar columnas con tipos incorrectos (por ejemplo, `price` como `object` en lugar de `float`).

In [ ]:
print('=== MADRID ===')
madrid.info()
print('\n=== MILÁN ===')
milan.info()

## 5. Estadísticas descriptivas

`describe()` nos da un resumen estadístico de las variables numéricas: media, desviación típica, mínimo, máximo y percentiles (25%, 50%, 75%). Nos ayuda a identificar rangos de valores, posibles outliers y la dispersión de cada variable.

In [ ]:
print('=== MADRID ===')
display(madrid.describe())
print('\n=== MILÁN ===')
display(milan.describe())

## 6. Valores nulos

Visualizamos la cantidad de valores nulos por columna. Identificar qué columnas tienen datos faltantes y en qué proporción es fundamental antes de cualquier análisis, ya que los nulos pueden sesgar los resultados o provocar errores en los modelos.

> En este notebook **detectamos** los nulos pero no los tratamos. La limpieza se realizará en un paso posterior.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, dataset, nombre in zip(axes, [madrid, milan], ['Madrid', 'Milán']):
    nulos = dataset.isnull().sum()
    nulos = nulos[nulos > 0]
    if nulos.empty:
        ax.text(0.5, 0.5, 'Sin valores nulos', ha='center', va='center', fontsize=13)
    else:
        nulos.plot(kind='bar', ax=ax, color='salmon', edgecolor='black')
    ax.set_title(f'Valores nulos — {nombre}')
    ax.set_ylabel('Número de nulos')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Distribución de variables numéricas

Analizamos visualmente cómo se distribuyen las principales variables numéricas. Las visualizaciones nos permiten detectar asimetrías, outliers y diferencias entre ciudades de un solo vistazo.

### 7.1 Distribución del precio (filtrando outliers)

Comparamos la distribución del precio por noche en Madrid y Milán. Filtramos el percentil 99 para eliminar valores extremos que distorsionarían la escala del gráfico. Esto es solo a efectos **visuales**, el dataset original no se modifica.

In [ ]:
# Filtramos outliers extremos (precio < percentil 99)
p99 = df['price'].quantile(0.99)
df_filt = df[df['price'] <= p99]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
sns.histplot(data=df_filt, x='price', hue='ciudad', kde=True,
             bins=50, ax=axes[0], palette=['#e63946', '#457b9d'])
axes[0].set_title('Distribución del precio por ciudad')
axes[0].set_xlabel('Precio (€/noche)')

# Boxplot
sns.boxplot(data=df_filt, x='ciudad', y='price', ax=axes[1],
            palette=['#e63946', '#457b9d'])
axes[1].set_title('Boxplot del precio por ciudad')
axes[1].set_ylabel('Precio (€/noche)')

plt.tight_layout()
plt.show()

print(df_filt.groupby('ciudad')['price'].describe().round(2))

### 7.2 Tipos de habitación

Airbnb clasifica los alojamientos en varias categorías: *Entire home/apt*, *Private room*, *Shared room* y *Hotel room*. Comparamos cómo se distribuyen en cada ciudad para entender la oferta disponible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, dataset, nombre, color in zip(
    axes, [madrid, milan], ['Madrid', 'Milán'], ['#e63946', '#457b9d']
):
    counts = dataset['room_type'].value_counts()
    counts.plot(kind='bar', ax=ax, color=color, edgecolor='black')
    ax.set_title(f'Tipos de habitación — {nombre}')
    ax.set_ylabel('Número de anuncios')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 7.3 Top 10 barrios por precio medio

Identificamos los barrios más caros de cada ciudad según el precio medio por noche. Esto nos da una idea de las zonas premium y puede ser relevante para análisis geoespaciales posteriores.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, dataset, nombre, color in zip(
    axes, [madrid, milan], ['Madrid', 'Milán'], ['#e63946', '#457b9d']
):
    top = (
        dataset.groupby('neighbourhood')['price']
        .mean()
        .sort_values(ascending=False)
        .head(10)
    )
    top.plot(kind='barh', ax=ax, color=color, edgecolor='black')
    ax.invert_yaxis()
    ax.set_title(f'Top 10 barrios (precio medio) — {nombre}')
    ax.set_xlabel('Precio medio (€/noche)')

plt.tight_layout()
plt.show()

### 7.4 Matriz de correlación (variables numéricas)

La matriz de correlación mide la relación lineal entre pares de variables numéricas (valores entre -1 y 1). Nos ayuda a detectar:
- Variables **muy correlacionadas** entre sí (posible redundancia)
- Variables con **relación con el precio** que podrían ser predictores útiles

In [ ]:
num_cols = ['price', 'minimum_nights', 'number_of_reviews',
            'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, dataset, nombre in zip(axes, [madrid, milan], ['Madrid', 'Milán']):
    corr = dataset[num_cols].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                ax=ax, linewidths=0.5, vmin=-1, vmax=1)
    ax.set_title(f'Correlación — {nombre}')

plt.tight_layout()
plt.show()